# GAEZ v5 - Suitability Class

No account needed. GAEZ v5 is actually distributed through a **public Google Cloud Storage bucket** (`gs://fao-gismgr-gaez-v5-data/`), not just through the FAO Hand-in-Hand web viewer (whose download button is, charitably, hard to find). Files are reachable over plain HTTPS, no `gsutil`/auth needed.

Reference: [GAEZ v5 Platform User Guide](https://github.com/un-fao/gaezv5/wiki/21.-GAEZ-v5-Platform-User-Guide) ("Download the data from Google Cloud Storage" section).

**Variable**: `RES05-SIX` = crop suitability index in classes (Suitability Class), for all land (not just existing cropland) - this is what the project plan calls for (S1-S4/marginal/not suitable classes).

**Config used** (kept the same across every crop so results stay comparable):
- Period: `HP0120` (2001-2020, most recent historical window - not a future projection)
- Climate data source: `AGERA5`
- SSP: `HIST` (historical, not a climate-change scenario)
- Water & input level: `HRLM` (rain-fed, high input level of management)

The files turned out to be tiny (~0.4-0.5 MB per crop, since it's a global categorical raster) - nowhere near the multi-GB file sizes I was originally worried about.

In [ ]:
# make sure we're running from scripts/download so `import common` resolves
import os, sys

NOTEBOOK_DIR = os.path.abspath(".")
if os.path.basename(NOTEBOOK_DIR) != "download":
    NOTEBOOK_DIR = os.path.join(NOTEBOOK_DIR, "scripts", "download")
os.chdir(NOTEBOOK_DIR)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)
print("cwd:", os.getcwd())

In [ ]:
from common import RAW_DIR, download_file, ensure_dir, get_logger

log = get_logger("gaez")

BASE_URL = "https://storage.googleapis.com/fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/MAPSET/RES05-SIX/"
OUT_DIR = RAW_DIR / "gaez_v5"

VARIABLE = "RES05-SIX"  # Crop suitability index in classes (Suitability Class), all land
PERIOD = "HP0120"  # 2001-2020
CLIMATE_SOURCE = "AGERA5"
SSP = "HIST"  # historical, not a climate-change scenario
WATER_INPUT = "HRLM"  # rain-fed, high input level of management

# GAEZ v5 crop code -> local label (see CODES_CROP in
# GAEZv5_Themes_Sub-themes_Variables.xlsx for more crop codes, e.g. WPO=potato, SUC=sugar cane)
CROPS = {
    "RCW": "rice",     # Wetland rice
    "MZE": "maize",
    "CSV": "cassava",
    "SOY": "soybean",
}


def build_filename(crop_code: str) -> str:
    return f"GAEZ-V5.{VARIABLE}.{PERIOD}.{CLIMATE_SOURCE}.{SSP}.{crop_code}.{WATER_INPUT}.tif"


def main() -> None:
    ensure_dir(OUT_DIR)
    for crop_code, label in CROPS.items():
        fname = build_filename(crop_code)
        url = BASE_URL + fname
        dest = OUT_DIR / f"suitability_class_{label}_{crop_code}.tif"
        try:
            download_file(url, dest)
        except Exception as e:  # noqa: BLE001
            log.warning("Failed to download %s (%s): %s", label, crop_code, e)

In [ ]:
main()

## Adding more crops

Look up the crop code in [GAEZv5_Themes_Sub-themes_Variables.xlsx](https://github.com/user-attachments/files/20961388/GAEZv5_Themes_Sub-themes_Variables.xlsx) (the `CODES_CROP` sheet), then add it to the `CROPS` dict above. E.g. `"WPO": "potato"` (white potato), `"SUC": "sugarcane"`.